<a href="https://colab.research.google.com/github/xrhd/mula/blob/refactor/yearly-project-layout/projects/2026/shrink_clip/Shrink_CLIP!_Knowledge_Distillation_in_JAX_Flax.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# Shrink CLIP: Knowledge Distillation in JAX/Flax

This notebook transfers part of the zero-shot knowledge of OpenAI's CLIP into a small convolutional neural network written with JAX and Flax NNX.

We compare two students on CIFAR-10:

1. **Baseline student:** trained with the hard CIFAR-10 labels.
2. **Distilled student:** trained with the hard labels plus CLIP's soft targets.

The experiment is educational rather than a benchmark. Distillation can improve calibration or robustness without improving clean accuracy on every run, so we will report the observed results instead of assuming that the distilled model always wins.


In [ ]:
# Colab uses the current kernel after this cell finishes.
!uv pip install --system -q jax flax optax transformers datasets torch torchvision treescope

In [ ]:
import jax
print(f"JAX version: {jax.__version__}")

try:
    import flax
    print(f"Flax version: {flax.__version__}")
except ImportError:
    print("Flax is not installed by default in this runtime.")

try:
    import optax
    print(f"Optax version: {optax.__version__}")
except ImportError:
    print("Optax is not installed by default in this runtime.")

## 0. Imports and setup

The memory setting must be applied before importing JAX. The teacher uses PyTorch and Hugging Face, while both student models use the same JAX/Flax implementation.


In [ ]:
import io
import os
import random
import time
import urllib.request

# Prevent JAX from reserving all accelerator memory before PyTorch loads CLIP.
os.environ.setdefault("XLA_PYTHON_CLIENT_PREALLOCATE", "false")

import jax
import jax.numpy as jnp
import matplotlib.animation as animation
import matplotlib.pyplot as plt
import numpy as np
import optax
import seaborn as sns
import torch
import torchvision.transforms as transforms
from datasets import load_dataset
from flax import nnx
from IPython.display import HTML
from PIL import Image
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader
from tqdm.auto import tqdm
from transformers import CLIPModel, CLIPProcessor

import treescope

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
treescope.basic_interactive_setup(autovisualize_arrays=True)
plt.style.use("dark_background")
print(f"Using PyTorch device: {DEVICE}")


In [ ]:
# Change these values to create shorter video-friendly runs.
TRAIN_DATA_LIMIT = -1  # Use the full 50,000-image training set.
NUM_EPOCHS = 30
BATCH_SIZE = 128
TEACHER_BATCH_SIZE = 64
LEARNING_RATE = 1e-3
TEMPERATURE = 3.0
ALPHA = 0.6  # Weight of the hard-label loss; 1 - ALPHA weights soft targets.
VALIDATION_FRACTION = 0.2
NOISE_LEVEL = 0.08

if TEMPERATURE <= 0:
    raise ValueError("TEMPERATURE must be positive.")
if not 0 <= ALPHA <= 1:
    raise ValueError("ALPHA must be between 0 and 1.")


## 1. A visual introduction to distillation

This illustrative animation shows a student's class distribution moving toward a teacher's soft distribution. It is a visual explanation, not a result from the trained models below.


In [ ]:
display_classes = [
    "airplane", "automobile", "bird", "cat", "deer",
    "dog", "frog", "horse", "ship", "truck",
]
x = np.arange(len(display_classes))
teacher_probs = np.array(
    [0.01, 0.02, 0.05, 0.65, 0.03, 0.20, 0.02, 0.01, 0.00, 0.01]
)
student_initial = np.array(
    [0.15, 0.10, 0.05, 0.05, 0.10, 0.10, 0.20, 0.10, 0.10, 0.05]
)

fig, (teacher_axis, student_axis) = plt.subplots(2, 1, figsize=(10, 7))
fig.suptitle(
    'Knowledge distillation: transferring "dark knowledge"',
    fontsize=16,
    fontweight="bold",
)

teacher_axis.bar(x, teacher_probs, color="#3b82f6", edgecolor="white")
teacher_axis.set_title("Teacher (CLIP): soft probabilities")
teacher_axis.set_xticks([])
teacher_axis.set_ylim(0, 1)
teacher_axis.grid(axis="y", alpha=0.2)

student_bars = student_axis.bar(
    x, student_initial, color="#ef4444", edgecolor="white"
)
student_axis.set_title("Student: moving toward the teacher")
student_axis.set_xticks(x)
student_axis.set_xticklabels(display_classes, rotation=30, ha="right")
student_axis.set_ylim(0, 1)
student_axis.grid(axis="y", alpha=0.2)
epoch_text = student_axis.text(
    0.84,
    0.85,
    "Illustration: 0",
    transform=student_axis.transAxes,
    fontsize=14,
    fontweight="bold",
    color="yellow",
)

def animate_intro(frame):
    progress = frame / 100.0
    ease = 1 - (1 - progress) ** 3
    current_probs = student_initial * (1 - ease) + teacher_probs * ease

    for bar, height in zip(student_bars, current_probs):
        bar.set_height(height)
        bar.set_color(
            (
                0.93 - (0.70 * ease),
                0.26 + (0.25 * ease),
                0.26 + (0.70 * ease),
            )
        )

    epoch_text.set_text(f"Illustration: {int(progress * 50)}")
    if frame == 100:
        student_axis.set_title(
            "Student: illustrative knowledge absorbed!",
            color="#86efac",
        )
    return student_bars, epoch_text

intro_animation = animation.FuncAnimation(
    fig,
    animate_intro,
    frames=101,
    interval=40,
    blit=False,
)
plt.close()
HTML(intro_animation.to_jshtml())


## 2. Load the CLIP teacher and CIFAR-10

CLIP is a contrastive image-text model. For this tutorial, its image-to-text similarity scores for ten class prompts become the teacher logits used to create soft targets.


In [ ]:
MODEL_NAME = "openai/clip-vit-base-patch16"
print(f"Loading CLIP teacher: {MODEL_NAME}")

teacher_model = CLIPModel.from_pretrained(MODEL_NAME).to(DEVICE)
teacher_model.eval()
processor = CLIPProcessor.from_pretrained(MODEL_NAME)

CIFAR10_CLASSES = [
    "airplane", "automobile", "bird", "cat", "deer",
    "dog", "frog", "horse", "ship", "truck",
]
TEXT_PROMPTS = [
    "a photo of an airplane",
    "a photo of an automobile",
    "a photo of a bird",
    "a photo of a cat",
    "a photo of a deer",
    "a photo of a dog",
    "a photo of a frog",
    "a photo of a horse",
    "a photo of a ship",
    "a photo of a truck",
]
transform = transforms.ToTensor()
NUM_CLASSES = len(CIFAR10_CLASSES)


In [ ]:
print("Downloading CIFAR-10 from the Hugging Face Hub...")
hf_dataset = load_dataset("uoft-cs/cifar10")
full_train = hf_dataset["train"]
test_subset = hf_dataset["test"]

if TRAIN_DATA_LIMIT == -1 or TRAIN_DATA_LIMIT >= len(full_train):
    train_subset = full_train
    print("Using the complete CIFAR-10 training set.")
elif 10 <= TRAIN_DATA_LIMIT < len(full_train):
    all_labels = np.asarray(full_train["label"])
    all_indices = np.arange(len(all_labels))
    subset_indices, _ = train_test_split(
        all_indices,
        train_size=TRAIN_DATA_LIMIT,
        stratify=all_labels,
        random_state=SEED,
    )
    train_subset = full_train.select(subset_indices)
    print(f"Using a balanced training subset of {TRAIN_DATA_LIMIT} images.")
else:
    raise ValueError(
        "TRAIN_DATA_LIMIT must be -1 or an integer between 10 and the "
        "full training-set size."
    )

print(f"Training images: {len(train_subset)}")
print(f"Test images: {len(test_subset)}")


## 3. Build the student model

The student is a small residual CNN. It receives the original 32x32 CIFAR-10 image, while CLIP receives its own resized and normalized representation of that same image.


In [ ]:
class ResidualBlock(nnx.Module):
    def __init__(
        self,
        in_features,
        out_features,
        stride=1,
        rngs: nnx.Rngs = None,
    ):
        self.conv1 = nnx.Conv(
            in_features=in_features,
            out_features=out_features,
            kernel_size=(3, 3),
            strides=(stride, stride),
            padding="SAME",
            rngs=rngs,
        )
        self.conv2 = nnx.Conv(
            in_features=out_features,
            out_features=out_features,
            kernel_size=(3, 3),
            padding="SAME",
            rngs=rngs,
        )
        if stride != 1 or in_features != out_features:
            self.shortcut = nnx.Conv(
                in_features=in_features,
                out_features=out_features,
                kernel_size=(1, 1),
                strides=(stride, stride),
                padding="SAME",
                rngs=rngs,
            )
        else:
            self.shortcut = lambda x: x

    def __call__(self, x):
        residual = self.shortcut(x)
        x = jax.nn.relu(self.conv1(x))
        x = self.conv2(x)
        return jax.nn.relu(x + residual)


class MiniResNet(nnx.Module):
    def __init__(self, rngs: nnx.Rngs):
        self.conv_init = nnx.Conv(
            in_features=3,
            out_features=32,
            kernel_size=(3, 3),
            padding="SAME",
            rngs=rngs,
        )
        self.layer1 = ResidualBlock(32, 32, rngs=rngs)
        self.layer2 = ResidualBlock(32, 64, stride=2, rngs=rngs)
        self.layer3 = ResidualBlock(64, 64, rngs=rngs)
        self.linear = nnx.Linear(
            in_features=64,
            out_features=NUM_CLASSES,
            rngs=rngs,
        )

    def __call__(self, x):
        x = jax.nn.relu(self.conv_init(x))
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = jnp.mean(x, axis=(1, 2))
        return self.linear(x)


## 4. Define the distillation loss

The baseline minimizes hard-label cross-entropy. The distilled student combines that loss with soft-target cross-entropy at a higher temperature. This soft-target term has the same student gradients as KL divergence, up to a teacher-only constant, and the usual temperature-squared correction keeps its scale comparable.


In [ ]:
def baseline_loss_fn(model, batch_images, batch_labels):
    logits = model(batch_images)
    one_hot = jax.nn.one_hot(batch_labels, NUM_CLASSES)
    loss = optax.softmax_cross_entropy(
        logits=logits,
        labels=one_hot,
    ).mean()
    return loss, logits


def distillation_loss_fn(
    model,
    batch_images,
    batch_labels,
    teacher_logits,
    temperature,
    alpha,
):
    student_logits = model(batch_images)
    one_hot = jax.nn.one_hot(batch_labels, NUM_CLASSES)
    hard_loss = optax.softmax_cross_entropy(
        logits=student_logits,
        labels=one_hot,
    ).mean()

    teacher_probs = jax.nn.softmax(teacher_logits / temperature)
    soft_loss = optax.softmax_cross_entropy(
        logits=student_logits / temperature,
        labels=teacher_probs,
    ).mean() * (temperature**2)

    loss = alpha * hard_loss + (1 - alpha) * soft_loss
    return loss, student_logits


@nnx.jit
def train_step_baseline(model, optimizer, batch_images, batch_labels):
    grad_fn = nnx.value_and_grad(baseline_loss_fn, has_aux=True)
    (loss, logits), grads = grad_fn(model, batch_images, batch_labels)
    optimizer.update(model, grads)
    return loss, logits


@nnx.jit
def train_step_distilled(
    model,
    optimizer,
    batch_images,
    batch_labels,
    teacher_logits,
    temperature,
    alpha,
):
    grad_fn = nnx.value_and_grad(distillation_loss_fn, has_aux=True)
    (loss, logits), grads = grad_fn(
        model,
        batch_images,
        batch_labels,
        teacher_logits,
        temperature,
        alpha,
    )
    optimizer.update(model, grads)
    return loss, logits


## 5. Precompute the teacher targets

CLIP is used only to produce targets. Precomputing those targets keeps the training loop entirely in JAX and avoids running the large teacher twice per training step.


In [ ]:
def precompute_teacher_outputs(dataset):
    image_batches = []
    label_batches = []
    logit_batches = []

    for start in tqdm(
        range(0, len(dataset), TEACHER_BATCH_SIZE),
        desc="Computing CLIP targets",
    ):
        batch_data = dataset[start : start + TEACHER_BATCH_SIZE]
        raw_images = batch_data["img"]
        labels = np.asarray(batch_data["label"], dtype=np.int32)

        with torch.inference_mode():
            inputs = processor(
                text=TEXT_PROMPTS,
                images=raw_images,
                return_tensors="pt",
                padding=True,
            ).to(DEVICE)
            teacher_logits = teacher_model(**inputs).logits_per_image

        student_images = torch.stack(
            [transform(image) for image in raw_images]
        )
        student_images = student_images.permute(0, 2, 3, 1).numpy()

        image_batches.append(student_images)
        label_batches.append(labels)
        logit_batches.append(teacher_logits.cpu().numpy())

    images = jnp.asarray(np.concatenate(image_batches), dtype=jnp.float32)
    labels = jnp.asarray(np.concatenate(label_batches), dtype=jnp.int32)
    logits = jnp.asarray(np.concatenate(logit_batches), dtype=jnp.float32)
    return images, labels, logits


jax_images_full, jax_labels_full, jax_logits_full = (
    precompute_teacher_outputs(train_subset)
)
print(f"Precomputed targets for {len(jax_labels_full)} images.")


In [ ]:
# Each row contains CLIP's ten class probabilities for one image.
jax.nn.softmax(jax_logits_full[:5]), jax_labels_full[:5]


## 6. Train and validate both students

The validation split is stratified and uses a fixed seed. Both students start from identical initial weights so the comparison focuses on the training objective.


In [ ]:
@nnx.jit
def eval_step(model, batch_images, batch_labels):
    logits = model(batch_images)
    one_hot = jax.nn.one_hot(batch_labels, NUM_CLASSES)
    loss = optax.softmax_cross_entropy(
        logits=logits,
        labels=one_hot,
    ).mean()
    return loss, logits


def evaluate_model(model, images, labels, batch_size):
    total_examples = int(len(labels))
    total_loss = 0.0
    total_correct = 0

    for start in range(0, total_examples, batch_size):
        batch_images = images[start : start + batch_size]
        batch_labels = labels[start : start + batch_size]
        loss, logits = eval_step(model, batch_images, batch_labels)
        count = len(batch_labels)
        total_loss += float(loss) * count
        total_correct += int(
            jnp.sum(jnp.argmax(logits, axis=-1) == batch_labels)
        )

    return total_loss / total_examples, total_correct / total_examples


split_indices, validation_indices = train_test_split(
    np.arange(len(jax_labels_full)),
    test_size=VALIDATION_FRACTION,
    stratify=np.asarray(jax_labels_full),
    random_state=SEED,
)
train_imgs = jax_images_full[split_indices]
train_lbls = jax_labels_full[split_indices]
train_logits = jax_logits_full[split_indices]
val_imgs = jax_images_full[validation_indices]
val_lbls = jax_labels_full[validation_indices]
val_logits = jax_logits_full[validation_indices]

# Separate RNG objects with the same seed give identical initial weights.
model_baseline = MiniResNet(nnx.Rngs(SEED))
model_distilled = MiniResNet(nnx.Rngs(SEED))
optimizer_baseline = nnx.Optimizer(
    model_baseline,
    optax.adam(LEARNING_RATE),
    wrt=nnx.Param,
)
optimizer_distilled = nnx.Optimizer(
    model_distilled,
    optax.adam(LEARNING_RATE),
    wrt=nnx.Param,
)

history = {
    "base_train_loss": [],
    "base_val_loss": [],
    "dist_train_loss": [],
    "dist_val_loss": [],
    "base_train_acc": [],
    "base_val_acc": [],
    "dist_train_acc": [],
    "dist_val_acc": [],
}

for epoch in range(NUM_EPOCHS):
    permutation = jax.random.permutation(
        jax.random.PRNGKey(SEED + epoch),
        len(train_lbls),
    )
    imgs_shuffled = train_imgs[permutation]
    lbls_shuffled = train_lbls[permutation]
    logits_shuffled = train_logits[permutation]

    base_loss_total = 0.0
    dist_loss_total = 0.0
    base_correct = 0
    dist_correct = 0
    total_examples = len(train_lbls)

    for start in tqdm(
        range(0, total_examples, BATCH_SIZE),
        desc=f"Epoch {epoch + 1}/{NUM_EPOCHS}",
        leave=False,
    ):
        batch_images = imgs_shuffled[start : start + BATCH_SIZE]
        batch_labels = lbls_shuffled[start : start + BATCH_SIZE]
        batch_teacher_logits = logits_shuffled[start : start + BATCH_SIZE]
        count = len(batch_labels)

        base_loss, base_logits = train_step_baseline(
            model_baseline,
            optimizer_baseline,
            batch_images,
            batch_labels,
        )
        dist_loss, dist_logits = train_step_distilled(
            model_distilled,
            optimizer_distilled,
            batch_images,
            batch_labels,
            batch_teacher_logits,
            TEMPERATURE,
            ALPHA,
        )

        base_loss_total += float(base_loss) * count
        dist_loss_total += float(dist_loss) * count
        base_correct += int(
            jnp.sum(jnp.argmax(base_logits, axis=-1) == batch_labels)
        )
        dist_correct += int(
            jnp.sum(jnp.argmax(dist_logits, axis=-1) == batch_labels)
        )

    base_val_loss, base_val_acc = evaluate_model(
        model_baseline,
        val_imgs,
        val_lbls,
        BATCH_SIZE,
    )
    dist_val_loss, dist_val_acc = evaluate_model(
        model_distilled,
        val_imgs,
        val_lbls,
        BATCH_SIZE,
    )

    history["base_train_loss"].append(base_loss_total / total_examples)
    history["dist_train_loss"].append(dist_loss_total / total_examples)
    history["base_train_acc"].append(base_correct / total_examples)
    history["dist_train_acc"].append(dist_correct / total_examples)
    history["base_val_loss"].append(base_val_loss)
    history["base_val_acc"].append(base_val_acc)
    history["dist_val_loss"].append(dist_val_loss)
    history["dist_val_acc"].append(dist_val_acc)

    print(
        f"Epoch {epoch + 1:02d}: "
        f"baseline val acc={base_val_acc:.4f}, "
        f"distilled val acc={dist_val_acc:.4f}"
    )


## 7. Inspect the training curves

The horizontal line is CLIP's zero-shot accuracy on the held-out validation split. It is a reference for this split, not a guarantee that a student can reach the teacher's accuracy.


In [ ]:
clip_preds = np.argmax(np.asarray(val_logits), axis=-1)
clip_acc = float(np.mean(clip_preds == np.asarray(val_lbls)))
print(f"CLIP zero-shot accuracy on validation: {clip_acc:.2%}")

epochs = np.arange(1, NUM_EPOCHS + 1)
sns.set_theme(style="darkgrid", context="talk")
fig, (loss_axis, accuracy_axis) = plt.subplots(1, 2, figsize=(16, 6))

sns.lineplot(
    x=epochs,
    y=history["base_train_loss"],
    ax=loss_axis,
    label="Baseline train",
    color="#fca5a5",
    linestyle="--",
)
sns.lineplot(
    x=epochs,
    y=history["base_val_loss"],
    ax=loss_axis,
    label="Baseline validation",
    color="#ef4444",
)
sns.lineplot(
    x=epochs,
    y=history["dist_train_loss"],
    ax=loss_axis,
    label="Distilled train",
    color="#93c5fd",
    linestyle="--",
)
sns.lineplot(
    x=epochs,
    y=history["dist_val_loss"],
    ax=loss_axis,
    label="Distilled validation",
    color="#3b82f6",
)
loss_axis.set_title("Loss")
loss_axis.set_xlabel("Epoch")
loss_axis.legend(fontsize=10)

sns.lineplot(
    x=epochs,
    y=history["base_train_acc"],
    ax=accuracy_axis,
    label="Baseline train",
    color="#fca5a5",
    linestyle="--",
)
sns.lineplot(
    x=epochs,
    y=history["base_val_acc"],
    ax=accuracy_axis,
    label="Baseline validation",
    color="#ef4444",
)
sns.lineplot(
    x=epochs,
    y=history["dist_train_acc"],
    ax=accuracy_axis,
    label="Distilled train",
    color="#93c5fd",
    linestyle="--",
)
sns.lineplot(
    x=epochs,
    y=history["dist_val_acc"],
    ax=accuracy_axis,
    label="Distilled validation",
    color="#3b82f6",
)
accuracy_axis.axhline(
    clip_acc,
    color="#eab308",
    linestyle="-.",
    label=f"CLIP validation ({clip_acc:.1%})",
)
accuracy_axis.set_title("Accuracy")
accuracy_axis.set_xlabel("Epoch")
accuracy_axis.set_ylim(0, 1)
accuracy_axis.legend(fontsize=10)
plt.tight_layout()
plt.show()


## 8. Evaluate on the unseen test set

This evaluation uses weighted totals, so the final partial batch is included correctly.


In [ ]:
def hf_collate(batch):
    images = torch.stack([transform(item["img"]) for item in batch])
    labels = torch.tensor([item["label"] for item in batch])
    return images, labels


test_loader = DataLoader(
    test_subset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=hf_collate,
)

test_image_batches = []
test_label_batches = []
for images, labels in tqdm(test_loader, desc="Preparing test set"):
    test_image_batches.append(
        images.numpy().transpose((0, 2, 3, 1))
    )
    test_label_batches.append(labels.numpy())

test_images = jnp.asarray(
    np.concatenate(test_image_batches),
    dtype=jnp.float32,
)
test_labels = jnp.asarray(
    np.concatenate(test_label_batches),
    dtype=jnp.int32,
)

base_test_loss, base_test_acc = evaluate_model(
    model_baseline,
    test_images,
    test_labels,
    BATCH_SIZE,
)
dist_test_loss, dist_test_acc = evaluate_model(
    model_distilled,
    test_images,
    test_labels,
    BATCH_SIZE,
)

print("Test accuracy")
print(f"Baseline:  {base_test_acc:.4f}")
print(f"Distilled: {dist_test_acc:.4f}")


## 9. Test robustness to Gaussian noise

This is one controlled corruption experiment. If the distilled model performs better here, that is evidence for this run and noise level, not a universal robustness claim.


In [ ]:
def add_gaussian_noise(images, noise_factor, seed):
    rng = np.random.default_rng(seed)
    image_array = np.asarray(images)
    noise = rng.normal(0.0, 1.0, size=image_array.shape)
    noisy_images = np.clip(
        image_array + noise_factor * noise,
        0.0,
        1.0,
    )
    return jnp.asarray(noisy_images, dtype=jnp.float32)


noisy_test_images = add_gaussian_noise(
    test_images,
    noise_factor=NOISE_LEVEL,
    seed=SEED,
)
base_noisy_loss, base_noisy_acc = evaluate_model(
    model_baseline,
    noisy_test_images,
    test_labels,
    BATCH_SIZE,
)
dist_noisy_loss, dist_noisy_acc = evaluate_model(
    model_distilled,
    noisy_test_images,
    test_labels,
    BATCH_SIZE,
)

print(f"Gaussian noise level: {NOISE_LEVEL}")
print(f"Baseline accuracy:  {base_noisy_acc:.4f}")
print(f"Distilled accuracy: {dist_noisy_acc:.4f}")

clean_values = [base_test_acc, dist_test_acc]
noisy_values = [base_noisy_acc, dist_noisy_acc]
labels = ["Baseline", "Distilled"]
x = np.arange(len(labels))
width = 0.35

fig, axis = plt.subplots(figsize=(8, 5))
axis.bar(x - width / 2, clean_values, width, label="Clean")
axis.bar(x + width / 2, noisy_values, width, label="Gaussian noise")
axis.set_ylabel("Accuracy")
axis.set_title("Clean versus noisy test accuracy")
axis.set_xticks(x)
axis.set_xticklabels(labels)
axis.set_ylim(0, 1)
axis.legend()
plt.show()

fig, axes = plt.subplots(1, 2, figsize=(8, 3))
axes[0].imshow(np.asarray(test_images[0]))
axes[0].set_title("Clean image")
axes[0].axis("off")
axes[1].imshow(np.asarray(noisy_test_images[0]))
axes[1].set_title(f"Noise level {NOISE_LEVEL}")
axes[1].axis("off")
plt.tight_layout()
plt.show()


## 10. Optional qualitative OOD examples

These internet images are outside CIFAR-10. The predictions are nearest-label guesses, not accuracy measurements. Network failures are skipped so this optional cell does not stop the tutorial.


In [ ]:
OOD_IMAGE_URLS = [
    "https://images.unsplash.com/photo-1561731216-c3a4d99437d5?w=512",
    "https://images.unsplash.com/photo-1456926631375-92c8ce872def?w=512",
    "https://images.unsplash.com/photo-1517849845537-4d257902454a?w=512",
    "https://images.unsplash.com/photo-1543466835-00a7907e9de1?w=512",
]

def download_rgb_image(url):
    request = urllib.request.Request(
        url,
        headers={"User-Agent": "Mozilla/5.0"},
    )
    with urllib.request.urlopen(request, timeout=30) as response:
        return Image.open(io.BytesIO(response.read())).convert("RGB")


original_images = []
loaded_images = []
for url in OOD_IMAGE_URLS:
    try:
        image = download_rgb_image(url)
        original_images.append(image)
        tensor = transforms.Resize((32, 32))(image)
        tensor = transforms.ToTensor()(tensor)
        loaded_images.append(tensor.permute(1, 2, 0).numpy())
    except Exception as error:
        print(f"Skipping {url}: {error}")

if not loaded_images:
    print("No OOD images were available.")
else:
    ood_batch = jnp.asarray(np.stack(loaded_images), dtype=jnp.float32)
    base_preds = np.asarray(
        jnp.argmax(model_baseline(ood_batch), axis=-1)
    )
    dist_preds = np.asarray(
        jnp.argmax(model_distilled(ood_batch), axis=-1)
    )

    fig, axes = plt.subplots(
        2,
        len(original_images),
        figsize=(4 * len(original_images), 8), # Increased height to accommodate the second row
    )

    axes = np.atleast_2d(axes).reshape(2, -1)
    for index in range(len(original_images)):
        axes[0, index].imshow(original_images[index])
        axes[0, index].axis("off")
        axes[0, index].set_title(
            f"Baseline: {CIFAR10_CLASSES[base_preds[index]]} | "
            f"Distilled: {CIFAR10_CLASSES[dist_preds[index]]}\n(Original)"
        )
        axes[1, index].imshow(loaded_images[index])
        axes[1, index].axis("off")
        axes[1, index].set_title("Resized (32x32)")

    plt.tight_layout()
    plt.show()

## 11. Compare model size and inference speed

The timing below measures model forward passes with already-prepared inputs. It includes CLIP's text encoder and excludes processor preprocessing, so it is a model-throughput comparison rather than an end-to-end application benchmark.


In [ ]:
clip_params = sum(parameter.numel() for parameter in teacher_model.parameters())
student_state = nnx.state(model_distilled)
student_params = sum(
    leaf.size for leaf in jax.tree_util.tree_leaves(student_state)
)

print("Parameter count")
print(f"CLIP teacher:   {clip_params:,}")
print(f"Student CNN:    {student_params:,}")
print(f"Parameter ratio: {clip_params / student_params:.0f}x")

sample_count = min(BATCH_SIZE, len(test_images))
dummy_images_jax = test_images[:sample_count]
dummy_images_pt = torch.from_numpy(
    np.asarray(dummy_images_jax).transpose((0, 3, 1, 2))
)
clip_inputs = processor(
    text=TEXT_PROMPTS,
    images=dummy_images_pt,
    return_tensors="pt",
    padding=True,
).to(DEVICE)

def synchronize_torch():
    if DEVICE.type == "cuda":
        torch.cuda.synchronize(DEVICE)


with torch.inference_mode():
    for _ in range(2):
        teacher_model(**clip_inputs)
    synchronize_torch()
    start_time = time.perf_counter()
    for _ in range(10):
        teacher_model(**clip_inputs)
    synchronize_torch()
    clip_time = (time.perf_counter() - start_time) / 10

@nnx.jit
def infer_student(model, images):
    return model(images)


infer_student(model_distilled, dummy_images_jax).block_until_ready()
start_time = time.perf_counter()
for _ in range(10):
    infer_student(model_distilled, dummy_images_jax).block_until_ready()
student_time = (time.perf_counter() - start_time) / 10

print(f"CLIP forward pass:    {clip_time * 1000:.2f} ms")
print(f"Student forward pass: {student_time * 1000:.2f} ms")
print(f"Measured speed ratio: {clip_time / student_time:.0f}x")


## 12. Optional video B-roll: validation predictions

This cell creates a 2x2 comparison of clean validation images and the three probability distributions. It is designed for screen recording and uses only images already held in memory.


In [ ]:
num_cols = min(2, len(val_imgs))
if num_cols == 0:
    raise ValueError("The validation split is empty.")
num_frames = max(1, min(15, len(val_imgs) // num_cols))

x_positions = np.arange(NUM_CLASSES)
width = 0.25
fig, axes = plt.subplots(2, num_cols, figsize=(8 * num_cols, 9))
axes = np.asarray(axes).reshape(2, num_cols)
plt.subplots_adjust(
    top=0.88,
    bottom=0.12,
    left=0.06,
    right=0.94,
    hspace=0.4,
    wspace=0.2,
)
fig.suptitle(
    "Baseline versus distilled student versus CLIP",
    fontsize=20,
    fontweight="bold",
)

image_plots = []
bar_plots = []
for column in range(num_cols):
    image_axis = axes[0, column]
    image_plot = image_axis.imshow(np.zeros((32, 32, 3)))
    image_axis.axis("off")
    image_plots.append(image_plot)

    bar_axis = axes[1, column]
    baseline_bars = bar_axis.bar(
        x_positions - width,
        np.zeros(NUM_CLASSES),
        width,
        label="Baseline",
        color="#94a3b8",
    )
    distilled_bars = bar_axis.bar(
        x_positions,
        np.zeros(NUM_CLASSES),
        width,
        label="Distilled",
        color="#ef4444",
    )
    teacher_bars = bar_axis.bar(
        x_positions + width,
        np.zeros(NUM_CLASSES),
        width,
        label="CLIP",
        color="#3b82f6",
    )
    bar_axis.set_xticks(x_positions)
    bar_axis.set_xticklabels(
        CIFAR10_CLASSES,
        rotation=35,
        ha="right",
        fontsize=10,
    )
    bar_axis.set_ylim(0, 1.05)
    if column == 0:
        bar_axis.legend(loc="upper right", fontsize=10)
    bar_plots.append((baseline_bars, distilled_bars, teacher_bars))


def animate_broll(frame):
    start = frame * num_cols
    batch_images = val_imgs[start : start + num_cols]
    batch_labels = val_lbls[start : start + num_cols]
    batch_teacher_logits = val_logits[start : start + num_cols]

    student_probs = jax.nn.softmax(
        model_distilled(batch_images),
        axis=-1,
    )
    baseline_probs = jax.nn.softmax(
        model_baseline(batch_images),
        axis=-1,
    )
    teacher_probs = jax.nn.softmax(batch_teacher_logits, axis=-1)
    artists = []

    for column in range(num_cols):
        image = batch_images[column]
        true_label = int(batch_labels[column])

        image_plots[column].set_data(image)
        axes[0, column].set_title(
            f"True label: {CIFAR10_CLASSES[true_label]}",
            color="#16a34a",
            fontweight="bold",
        )
        artists.append(image_plots[column])
        artists.append(axes[0, column].title)

        # Highlight the correct class on the x-axis.
        tick_labels = axes[1, column].get_xticklabels()
        for class_index, tick in enumerate(tick_labels):
            is_correct_class = class_index == true_label
            tick.set_color(
                "#16a34a" if is_correct_class else "#374151"
            )
            tick.set_fontweight(
                "bold" if is_correct_class else "normal"
            )
            tick.set_fontsize(13 if is_correct_class else 10)
        artists.extend(tick_labels)

        baseline_bars, distilled_bars, teacher_bars = bar_plots[column]
        for class_index in range(NUM_CLASSES):
            is_correct_class = class_index == true_label
            edge_color = "#facc15" if is_correct_class else "white"
            edge_width = 3 if is_correct_class else 0.5

            bars = (
                baseline_bars[class_index],
                distilled_bars[class_index],
                teacher_bars[class_index],
            )
            for bar in bars:
                bar.set_edgecolor(edge_color)
                bar.set_linewidth(edge_width)
                artists.append(bar)

            baseline_bars[class_index].set_height(
                float(baseline_probs[column, class_index])
            )
            distilled_bars[class_index].set_height(
                float(student_probs[column, class_index])
            )
            teacher_bars[class_index].set_height(
                float(teacher_probs[column, class_index])
            )

    return artists


broll_animation = animation.FuncAnimation(
    fig,
    animate_broll,
    frames=num_frames,
    interval=2500,
    blit=False,
)
plt.close()
HTML(broll_animation.to_jshtml())

## Takeaways

- CLIP provides richer targets than a one-hot label, but the teacher is much larger and slower.
- A distilled student is not guaranteed to beat the baseline on clean accuracy.
- Robustness findings should be reported with the corruption type, noise level, seed, and dataset split.
- The most useful result for a tutorial is a transparent comparison that viewers can rerun and inspect.
